In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, Concatenate, Dropout, Bidirectional
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
# PARAMETERS
data_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\1_Friends_processed_featured.csv"
test_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\datasets\1_testing_dataset_g21_mapped_processed_featured.csv"
glove_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\glove.6B.300d.txt"
max_words = 10000
max_len = 100
pos_max_len = 100
embedding_dim = 300

In [4]:
df = pd.read_csv(data_path)
test_df = pd.read_csv(test_path)

In [5]:
print("Training data shape:", df.shape)
print("Test data shape:", test_df.shape)
print("\nTraining data columns:", df.columns.tolist())
print("\nTest data columns:", test_df.columns.tolist())

Training data shape: (9989, 20)
Test data shape: (860, 19)

Training data columns: ['Unnamed: 0', 'text', 'happiness', 'sadness', 'disgust', 'anger', 'fear', 'surprise', 'neutral', 'POS_Tags', 'TF_IDF', 'TextBlob_Polarity', 'TextBlob_Subjectivity', 'TextBlob_Sentiment', 'VADER_Scores', 'VADER_Compound', 'VADER_Positive', 'VADER_Neutral', 'VADER_Negative', 'TextBlob_Scores']

Test data columns: ['text', 'POS_Tags', 'TF_IDF', 'TextBlob_Polarity', 'TextBlob_Subjectivity', 'TextBlob_Sentiment', 'VADER_Scores', 'VADER_Compound', 'VADER_Positive', 'VADER_Neutral', 'VADER_Negative', 'TextBlob_Scores', 'happiness', 'sadness', 'disgust', 'anger', 'fear', 'surprise', 'neutral']


In [6]:
# Create target label by taking argmax across emotion columns
emotion_cols = ['happiness', 'sadness', 'disgust', 'anger', 'fear', 'surprise', 'neutral']
df['label'] = df[emotion_cols].values.argmax(axis=1)
test_df['label'] = test_df[emotion_cols].values.argmax(axis=1)
num_classes = len(emotion_cols)

# One-hot encode labels
y = to_categorical(df['label'], num_classes=num_classes)
y_test = to_categorical(test_df['label'], num_classes=num_classes)

# Prepare sentiment features
sentiment_features = df[['TextBlob_Polarity', 'VADER_Compound']].values
test_sentiment_features = test_df[['TextBlob_Polarity', 'VADER_Compound']].values

# Normalize sentiment features
scaler_sentiment = StandardScaler()
sentiment_features = scaler_sentiment.fit_transform(sentiment_features)
test_sentiment_features = scaler_sentiment.transform(test_sentiment_features)

In [7]:
# Tokenize text
df['text'] = df['text'].astype(str)
test_df['text'] = test_df['text'].astype(str)

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])
test_sequences = tokenizer.texts_to_sequences(test_df['text'])

X_text = pad_sequences(sequences, maxlen=max_len)
X_text_test = pad_sequences(test_sequences, maxlen=max_len)

# Process POS tags
df['POS_Tags'] = df['POS_Tags'].astype(str)
test_df['POS_Tags'] = test_df['POS_Tags'].astype(str)

pos_tokenizer = Tokenizer(num_words=5000)  # Fewer unique tokens for POS tags
pos_tokenizer.fit_on_texts(df['POS_Tags'])
pos_sequences = pos_tokenizer.texts_to_sequences(df['POS_Tags'])
test_pos_sequences = pos_tokenizer.texts_to_sequences(test_df['POS_Tags'])

X_pos = pad_sequences(pos_sequences, maxlen=pos_max_len)
X_pos_test = pad_sequences(test_pos_sequences, maxlen=pos_max_len)

print(f"Text vocabulary size: {len(tokenizer.word_index)}")
print(f"POS tags vocabulary size: {len(pos_tokenizer.word_index)}")

Text vocabulary size: 5794
POS tags vocabulary size: 5863


In [8]:
# Split into train and validation
X_text_train, X_text_val, X_pos_train, X_pos_val, X_sent_train, X_sent_val, y_train, y_val = train_test_split(
    X_text, X_pos, sentiment_features, y, test_size=0.20, random_state=42, stratify=df['label'])


In [9]:
# LOAD PRETRAINED GLOVE EMBEDDINGS
embeddings_index = {}
with open(glove_path, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coeffs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coeffs
print(f'Found {len(embeddings_index)} word vectors in GloVe.')

Found 400000 word vectors in GloVe.


In [10]:
# CREATE EMBEDDING MATRIX
word_index = tokenizer.word_index
num_words = min(max_words, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if i >= max_words:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [11]:
# Text branch with increased regularization
text_input = Input(shape=(max_len,), name='text_input')
embedding_layer = Embedding(input_dim=num_words,
                           output_dim=embedding_dim,
                           weights=[embedding_matrix],
                           input_length=max_len,
                           trainable=False)(text_input)
# Add recurrent dropout back and reduce units to prevent overfitting
lstm_out = Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3, 
                             return_sequences=True))(embedding_layer)
# Add another LSTM layer to create deeper context understanding
lstm_out = Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3))(lstm_out)
lstm_out = Dropout(0.4)(lstm_out)  # Extra dropout after LSTM

# POS tags branch with increased regularization
pos_input = Input(shape=(pos_max_len,), name='pos_input')
pos_embedding = Embedding(input_dim=len(pos_tokenizer.word_index) + 1,
                         output_dim=50,
                         input_length=pos_max_len)(pos_input)
pos_lstm = Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3))(pos_embedding)
pos_lstm = Dropout(0.4)(pos_lstm)  # Extra dropout after LSTM

# Sentiment features branch with L2 regularization
sent_input = Input(shape=(sentiment_features.shape[1],), name='sentiment_input')
sent_dense = Dense(32, activation='relu', 
                  kernel_regularizer=tf.keras.regularizers.l2(0.001))(sent_input)
sent_dense = Dropout(0.4)(sent_dense)

# Combine all three branches
combined = Concatenate()([lstm_out, pos_lstm, sent_dense])
combined_dense = Dense(64, activation='relu', 
                      kernel_regularizer=tf.keras.regularizers.l2(0.001))(combined)
combined_dense = Dropout(0.5)(combined_dense)
# Add another dense layer with batch normalization for better feature processing
combined_dense = Dense(32, activation='relu')(combined_dense)
combined_dense = tf.keras.layers.BatchNormalization()(combined_dense)
combined_dense = Dropout(0.5)(combined_dense)
output = Dense(num_classes, activation='softmax')(combined_dense)

# Build and compile the model with reduced learning rate
model = Model(inputs=[text_input, pos_input, sent_input], outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),  # Reduced learning rate
             loss='categorical_crossentropy',
             metrics=['accuracy'])

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 text_input (InputLayer)        [(None, 100)]        0           []                               
                                                                                                  
 embedding (Embedding)          (None, 100, 300)     1738500     ['text_input[0][0]']             
                                                                                                  
 pos_input (InputLayer)         [(None, 100)]        0           []                               
                                                                                                  
 bidirectional (Bidirectional)  (None, 100, 128)     186880      ['embedding[0][0]']              
                                                                                              

In [14]:
# Define callbacks with improved early stopping and learning rate reduction
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001),
    ModelCheckpoint(filepath='best_model_friends.h5', monitor='val_loss', save_best_only=True)
]

# Train the model
history = model.fit(
    {'text_input': X_text_train, 'pos_input': X_pos_train, 'sentiment_input': X_sent_train},
    y_train,
    epochs=150,
    batch_size=32,
    validation_data=({'text_input': X_text_val, 'pos_input': X_pos_val, 'sentiment_input': X_sent_val}, y_val),
    callbacks=callbacks
)

Epoch 1/150
250/250 [==============================] - 976s 4s/step - loss: 2.4145 - accuracy: 0.2253 - val_loss: 1.7469 - val_accuracy: 0.4735 - lr: 5.0000e-04
Epoch 2/150
250/250 [==============================] - 957s 4s/step - loss: 1.9433 - accuracy: 0.3763 - val_loss: 1.6412 - val_accuracy: 0.4715 - lr: 5.0000e-04
Epoch 3/150
250/250 [==============================] - 966s 4s/step - loss: 1.7459 - accuracy: 0.4524 - val_loss: 1.6301 - val_accuracy: 0.4715 - lr: 5.0000e-04
Epoch 4/150
250/250 [==============================] - 960s 4s/step - loss: 1.6600 - accuracy: 0.4704 - val_loss: 1.6160 - val_accuracy: 0.4715 - lr: 5.0000e-04
Epoch 5/150
250/250 [==============================] - 934s 4s/step - loss: 1.6357 - accuracy: 0.4720 - val_loss: 1.6048 - val_accuracy: 0.4715 - lr: 5.0000e-04
Epoch 6/150
250/250 [==============================] - 904s 4s/step - loss: 1.6224 - accuracy: 0.4759 - val_loss: 1.5978 - val_accuracy: 0.4715 - lr: 5.0000e-04
Epoch 7/150
250/250 [=============

KeyboardInterrupt: 

In [ ]:
# Evaluate on test set
loss, acc = model.evaluate(
    {'text_input': X_text_test, 'pos_input': X_pos_test, 'sentiment_input': test_sentiment_features}, 
    y_test
)
print(f'Test Loss: {loss:.4f} / Test Accuracy: {acc:.4f}')

# Generate predictions
y_pred = model.predict(
    {'text_input': X_text_test, 'pos_input': X_pos_test, 'sentiment_input': test_sentiment_features}
)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred_classes)
precision = precision_score(y_true, y_pred_classes, average='weighted')
recall = recall_score(y_true, y_pred_classes, average='weighted')
f1 = f1_score(y_true, y_pred_classes, average='weighted')


In [ ]:
# Print metrics
print("\nFriends Dataset - Performance on Test Set:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")



In [ ]:
# Print detailed classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes))

In [ ]:
# Plot Loss Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracy Curves
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix - Friends Dataset')
plt.show()

In [ ]:
# Save model
model.save("lstm_model_friends.h5")